# Business Intelligence Workshop 

https://github\.com/ians513/Business\-Intelligence

Authors:

- Juan Pablo Molina

- Ian Spikin

# W1\-A · Relevance and documentary foundation — 25 points

Core evidence: at least two meaningfully distinct candidate problems with relevant, retrievable documentary
 support for each

Candidate A — Sales behavior by region:
How do the best\-selling product categories vary across regions, and what factors are associated with lower sales volumes?

The goal is to increase sales volume by implementing an appropriate strategy, either by promoting frequently purchased products or by investigating the least\-selling products to determine whether there are barriers affecting their sales or whether the market for them is smaller\.

Candidate B — Logistics efficiency and delays:
How does the risk of late delivery vary across regions and product categories, and what operational factors are associated with these differences?

The goal is to identify the products with the highest number of late deliveries and the regions experiencing the most problems in order to take appropriate action\.

Source used to verify and analyze the problems:

https://data\.mendeley\.com/datasets/8gx2fvg2k6/5

Bibliographic Sources:

What is the value of delivering on time?

https://www\.emerald\.com/jamr/article\-abstract/17/4/473/201694/What\-is\-the\-value\-of\-delivering\-on\-time?redirectedFrom=fulltext

The Effect of Delivery Time on Repurchase Behavior in Quick Commerce

https://www\.researchgate\.net/publication/384990655\_The\_Effect\_of\_Delivery\_Time\_on\_Repurchase\_Behavior\_in\_Quick\_Commerce

# W1\-B · Actual data inspection and initial understanding — 30 points

Core evidence: source\-specific inspection for both lines or a documented infeasibility finding for one; at least one
actual\-data summary with interpretation; identifiable observation units and coverage for the inspected data\.

The dataset obtained from the source mentioned in Section A is reviewed: DataCoSupplyChainDataset\.csv

To facilitate the work, the dataset is converted to Parquet format, which stores the DataFrame more compactly, allowing it to be read and imported faster\.

The DataCo Supply Chain dataset contains 180,519 records and 53 variables related to sales, products, customers, orders, and logistics processes\. It includes information on product categories, regions, shipping methods, actual and scheduled delivery times, order status, and late delivery risk\. This allows us to analyze sales patterns and study possible factors associated with late deliveries\.

In [ ]:
import pandas as pd

The CSV-to-Parquet conversion is run once, and the Parquet file is then used to load the dataset in future sessions.

In [ ]:
# df = pd.read_csv("/work/DataCoSupplyChainDataset.csv", encoding = "ISO-8859-1")
# df = df.convert_dtypes()
# df.to_parquet('data.parquet')
df = pd.read_parquet('data.parquet')

In [ ]:
df

Dataset data and characteristics

In [ ]:
# Dataset dimensions
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Data types
display(df.dtypes)

# Missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
display(missing_values[missing_values > 0])

# Duplicates
print("Duplicate rows:", df.duplicated().sum())


Candidate A:
“How do the best-selling product categories vary across regions, and what factors are associated with lower sales volumes?”

To answer the question, the following dataset variables will be used primarily:

Order Region: region associated with the order.
Category Name: category to which the product belongs.
Order Item Quantity: number of units sold of a product.
Sales: sales value associated with the product or order.
Order Item Product Price: unit price of the product.
Order Item Discount: discount amount applied to the product.
Order Item Discount Rate: discount percentage applied.
Customer Segment: segment to which the customer belongs.
Order Id: order identifier.
Order Item Id: identifier of the product within the order.

In [ ]:
columns = [
    "Order Region",
    "Category Name",
    "Order Item Quantity",
    "Sales",
    "Order Item Product Price",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Customer Segment",
    "Order Id",
    "Order Item Id"
]

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df[columns].head())
display(df[columns].isnull().sum())


In [ ]:
sales_by_region = (
    df.groupby("Order Region")["Order Item Quantity"]
      .agg(["sum", "count"])
      .sort_values("sum", ascending=False)
)

sales_by_region.columns = [
    "Units sold",
    "Number of records"
]

display(sales_by_region)


In [ ]:
sales_by_category = (
    df.groupby("Category Name")["Order Item Quantity"]
      .agg(["sum", "count"])
      .sort_values("sum", ascending=False)
)

sales_by_category.columns = [
    "Units sold",
    "Number of records"
]

display(sales_by_category)


In [ ]:
sales_by_region_category = (
    df.groupby(["Order Region", "Category Name"])["Order Item Quantity"]
      .sum()
      .reset_index()
      .sort_values(
          ["Order Region", "Order Item Quantity"],
          ascending=[True, False]
      )
)

top_categories_by_region = (
    sales_by_region_category
    .groupby("Order Region")
    .head(5)
)

display(top_categories_by_region)


In [ ]:
sales_price = (
    df.groupby("Category Name")
      .agg(
          units_sold=("Order Item Quantity", "sum"),
          average_price=("Order Item Product Price", "mean")
      )
      .sort_values("units_sold", ascending=False)
)

sales_price


In [ ]:
sales_discount = (
    df.groupby("Category Name")
      .agg(
          units_sold=("Order Item Quantity", "sum"),
          average_discount=("Order Item Discount Rate", "mean")
      )
      .sort_values("units_sold", ascending=False)
)

display(sales_discount)


The results make it possible to compare whether there are differences in sales volume across regions and product categories. In addition, possible associations between sales volume and factors such as price and applied discounts can be observed. These differences should initially be interpreted as associations rather than causal relationships, since other factors, such as customer segment or the specific characteristics of each region, could also have an influence.

Candidate B
“How does the risk of late delivery vary across regions and product categories, and what operational factors are associated with these differences?”

To answer the question, the following dataset variables will be used primarily:

Late_delivery_risk: indicates whether a delivery presents a risk of delay.
Order Region: allows delivery behavior to be compared across different regions.
Category Name: identifies the product category associated with the order.
Shipping Mode: represents the shipping method used.
Days for shipping (real): actual number of days taken for shipping.
Days for shipment (scheduled): number of days scheduled for shipping.

In [ ]:
columns = [
    "Late_delivery_risk",
    "Order Region",
    "Category Name",
    "Shipping Mode",
    "Days for shipping (real)",
    "Days for shipment (scheduled)"
]

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df[columns].head())
display(df[columns].isnull().sum())


In [ ]:
risk_by_region = (
    df.groupby("Order Region")["Late_delivery_risk"]
      .agg(["mean", "count"])
      .sort_values("mean", ascending=False)
)

risk_by_region["mean"] *= 100

risk_by_region.columns = [
    "Late delivery risk (%)",
    "Number of records"
]

display(risk_by_region)


In [ ]:
import matplotlib.pyplot as plt

# Late-delivery risk by region chart
ax = risk_by_region["Late delivery risk (%)"].plot(
    kind="bar",
    figsize=(10, 5),
)

ax.set_title("Late Delivery Risk by Region")
ax.set_ylim(40, 60)
ax.set_ylabel("Late Delivery Risk (%)")
ax.set_xlabel("Region")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

plt.close()


In [ ]:
risk_by_shipping_mode = (
    df.groupby("Shipping Mode")["Late_delivery_risk"]
      .agg(["mean", "count"])
      .sort_values("mean", ascending=False)
)

risk_by_shipping_mode["mean"] *= 100

display(risk_by_shipping_mode)


The results make it possible to observe whether there are differences in late-delivery risk across regions and shipping methods. These variations may be associated with operational factors, although at this stage a causal relationship cannot be established.

# W1\-C · Critical comparison and questions for feedback — 30 points

Core evidence: an evidence\-based comparison of the alternatives, a provisional preference and a concrete
uncertainty on which feedback is requested\.

Both candidates present a clear opportunity for analysis, although they have different approaches. Candidate A focuses on sales behavior, while Candidate B focuses on the risk of order delays.

| Categories | Candidate A | Candidate B |
| --- | --- | --- |
| Objective | Analyze sales differences across regions and categories. | Analyze delay risk by region, category, and shipping method. |
| Available data | Region, category, sales, price, and discounts. | Delay risk, region, category, shipping method, and delivery times. |
| Opportunity | Identify commercial patterns and factors associated with sales. | Identify factors associated with a higher probability of delay. |
| Limitation | Sales differences may depend on external factors that are not available in the dataset. | It is necessary to verify how `Late_delivery_risk` is constructed and which variables are available before delivery. |
| Future ML | Sales prediction. | Delay-risk classification. |
| Dashboard | Sales by region, category, and product. | Delay risk by region, category, and shipping method. |

We selected Candidate A because it allows us to analyze historical sales behavior across regions and categories using different variables available in the dataset. This would make it possible to identify products with higher and lower demand and analyze associated factors such as price and discounts. In addition, it has potential for the later development of a sales prediction model and a commercial dashboard.
Our choice could change if it is determined that the available variables do not sufficiently explain sales differences across regions.

For candidate B, there's some interesting initial findings, with two main outliers: Canada and Central Africa, as Canada has a lower risk of a late delivery compared to other regions, whereas Central Africa has an overall risk higher than the mean, with the difference between these two regions being around 10pp\.

Despite these findings, the surrounding variables are hardly enough to investigate this problem further\. While a small breakdown can be made, comparing the differences in delivery times for all 4 categories, per region, further investigation is limited, since neither the current data nor ML techniques can explain and help solve the issue of lateness\. Most of the variables hold data about the products or the clients, and focusing all of the analysis and investigation on these variables will create a shallow, not in\-depth analysis\. On the other hand, while Candidate A's initial focus is to identify best and worst selling products, we identify the potential for a deeper analysis, as there's richer data about the products, since it can be paired with the clients', which can ultimately lead into the training of an ML algorithm that could spot complex patterns not apparent at first glance\.

Questions for feedback:
Does Candidate A provide sufficient depth and analytical opportunities to continue developing it throughout the project?

Would it be more appropriate to keep the sales analysis considering both region and category, or to focus mainly on one of these dimensions?

# W1\-D · Traceability, contributions and AI accountability — 15 points

An in\-depth report and all the relevant data and analysis can be found at: 
https://github\.com/ians513/Business\-Intelligence

Decisions and Checks
Two candidates were analyzed using the available variables and initial results\. Candidate A was provisionally selected because of its potential to analyze sales patterns across regions and categories\.

Individual Contributions
Ian Spikin and Juan Pablo: Both members worked together on the candidate definition, data exploration, analysis, interpretation of results, comparison of alternatives, and development of the notebook\.

Individual Reflection
Both members participated throughout the complete analysis\. This activity helped us understand how data exploration can be used to compare analytical questions and identify the opportunities and limitations of a dataset\.

AI Use
ChatGPT was used to support writing and organization\. Its suggestions were reviewed and adapted, while numerical results were verified directly using Python and the dataset\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5db68b16-0b39-495e-8ed7-dcd14b08540e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>